In [1]:
import warnings
from pathlib import Path

import numpy as np
import prism

from imagematerials import buildings as bld
from imagematerials import vehicles as vhc
from imagematerials.concepts import create_building_graph, create_vehicle_graph
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import GenericMaterials, GenericStocks, MaterialIntensities
from imagematerials.util import (
    export_to_netcdf,
    import_from_netcdf,
    read_circular_economy_config,
    read_climate_policy_config,
    rebroadcast_prep_data,
)


In [2]:
def get_vema_prep():
    base_dir = Path("..", "data", "raw")
    prep_fp = Path("prep_vema.nc")
    if not prep_fp.is_file():
        climate_policy_scenario_dir = base_dir / 'SSP2'
        circular_economy_scenario_dirs = {"slow": base_dir / 'circular_economy_scenarios' / 'slow'}

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
            circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
            prep_data = vhc.preprocess(base_dir, climate_policy_config, circular_economy_config)

        export_to_netcdf(prep_data, prep_fp)

    prep_data = import_from_netcdf(prep_fp)
    share_coords = set()
    for cur_type in prep_data["shares"].Type.values:
        share_coords.add(cur_type.split(" - ")[0])
    output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
    prep_data.pop("shares")
    knowledge_graph = create_vehicle_graph()
    new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
    region_coords = np.sort(prep_data["stocks"].coords["Region"].values.astype(int)).astype(str)[:-2]
    new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=region_coords)
    new_prep_data["knowledge_graph"] = knowledge_graph

    new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")
    new_prep_data["shares"] = None
    sec_vhc = Sector("vhc", new_prep_data)
    return sec_vhc

In [3]:
def get_buildings_prep():
    base_dir = Path("..", "IMAGE-Mat_old_version", "IMAGE-Mat", "BUMA")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = bld.preprocess(base_dir)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    new_prep_data["shares"] = None
    sec_bld = Sector("bld", new_prep_data)
    return sec_bld

In [4]:
sectors = [get_vema_prep(), get_buildings_prep()]
ns_coordinates = {
    "Region": sectors[0].coordinates["Region"],
    "material": list(set(sectors[0].coordinates["material"] + sectors[1].coordinates["material"]))
}
ns_combined = Sector("combi", {}, coordinates=ns_coordinates)
sectors.append(ns_combined)

In [5]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [6]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")

@prism.interface
class SumMaterials(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("inflow_materials", )
    output_data: tuple[str] = ("summed_inflow_materials", )



    summed_inflow_materials: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Time, inflow_materials):
        t, dt = time.t, time.dt
        self.summed_inflow_materials[t].loc[:] = 0
        for inflow in inflow_materials:
            self.summed_inflow_materials[t].loc[{"material": inflow[t].coords["material"]}] += inflow[t].sum("Type")

In [7]:
factory = ModelFactory(
    sectors, complete_timeline
    ).add(GenericStocks, ["bld", "vhc"]
    ).add(GenericMaterials,  "vhc"
    ).add(MaterialIntensities, "bld",
    ).add(SumMaterials, "combi", input_sources={"inflow_materials": ["vhc", "bld"]}
    )
model = factory.finish()

In [8]:
factory.visualize(px_x="1000px", px_y="700px").show("model.html", notebook=True)

model.html


In [9]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(complete_timeline)

In [10]:
model.combi["summed_inflow_materials"][2020]

<xarray.DataArray (Region: 26, material: 18)> Size: 4kB
<Quantity([[5.52835774e+04 1.09638353e+08 0.00000000e+00 1.63885476e+09
  0.00000000e+00 1.42805071e+05 1.04977295e+07 0.00000000e+00
  0.00000000e+00 3.28308921e+00 1.67020792e+08 0.00000000e+00
  0.00000000e+00 0.00000000e+00 1.25691690e+08 8.86467719e+01
  1.19001852e+07 2.10935848e+04]
 [1.73750957e+05 2.78532264e+09 0.00000000e+00 5.02736328e+10
  0.00000000e+00 2.44174808e+06 3.52066420e+08 0.00000000e+00
  0.00000000e+00 7.79989420e+01 5.18745999e+09 0.00000000e+00
  0.00000000e+00 0.00000000e+00 3.83276350e+09 3.19379384e+02
  2.94309303e+08 9.60175131e+04]
 [6.76575850e+04 4.29369548e+08 0.00000000e+00 2.23178920e+09
  0.00000000e+00 7.71291851e+04 1.23707250e+07 0.00000000e+00
  0.00000000e+00 0.00000000e+00 3.24986889e+08 0.00000000e+00
  0.00000000e+00 0.00000000e+00 1.51396667e+08 1.10868099e+02
  5.17549038e+07 2.63326466e+04]
 [3.44946666e+04 2.91472961e+08 0.00000000e+00 4.84168073e+09
  0.00000000e+00 1.93650048e+05 3.00265234e+07 0.00000000e+00
  0.00000000e+00 0.00000000e+00 6.16310998e+08 0.00000000e+00
  0.00000000e+00 0.00000000e+00 3.79507113e+08 5.56878741e+01
  3.73305947e+07 1.35724337e+04]
...
 [6.26118485e+04 2.02003924e+09 0.00000000e+00 8.18038173e+09
  0.00000000e+00 4.37877561e+07 5.60396013e+08 2.72027850e+05
  1.09792334e+07 4.55538678e+00 1.58528678e+09 0.00000000e+00
  0.00000000e+00 5.14559212e+07 4.70673062e+08 1.52341672e+02
  3.59310117e+08 3.43796155e+04]
 [2.86504926e+04 1.68168009e+08 0.00000000e+00 1.44424828e+09
  0.00000000e+00 2.83905076e+05 8.30059112e+06 0.00000000e+00
  0.00000000e+00 3.35888348e+01 2.15617702e+08 0.00000000e+00
  0.00000000e+00 0.00000000e+00 1.03949619e+08 4.91320569e+01
  2.15317164e+07 1.36727814e+04]
 [1.33623554e+05 1.71927168e+09 0.00000000e+00 7.02799430e+09
  0.00000000e+00 7.62048605e+04 3.87519892e+07 0.00000000e+00
  0.00000000e+00 4.52177209e+00 1.04565548e+09 0.00000000e+00
  0.00000000e+00 0.00000000e+00 4.54748090e+08 1.61353600e+02
  1.97173674e+08 4.43109957e+04]
 [6.99491698e+04 7.69865928e+08 0.00000000e+00 3.52483187e+09
  0.00000000e+00 5.27846450e+06 8.19215658e+07 3.23312177e+04
  1.30491045e+06 2.93490913e+00 5.65738469e+08 0.00000000e+00
  0.00000000e+00 6.11567012e+06 2.21041678e+08 9.45461308e+01
  1.05937739e+08 2.45022199e+04]], 'count')>
Coordinates:
  * Region    (Region) <U2 208B '1' '2' '3' '4' '5' ... '22' '23' '24' '25' '26'
  * material  (material) <U9 648B 'Concrete' 'Aluminium' ... 'Glass' 'Brick'